# BrowseMind AI Provider Test Notebook

This notebook is designed to test the AI connectivity exactly as implemented in the BrowseMind extension.

It covers two scenarios:
1. **Local Ollama (Offline)**: Uses the `/api/generate` endpoint without an API key.
2. **OpenAI Compatible API (Online/Keyed)**: Uses the `/chat/completions` endpoint with a Bearer token.

In [1]:
import requests
import json
import time

def print_result(title, result):
    print(f"\n{'='*20} {title} {'='*20}")
    print(result)

## 1. Local Ollama Test (Offline)
This mimics `src/ai/ollama.ts`. It connects to a local instance and uses the generate API.

In [ ]:
import time
import requests

# --- Configuration ---
OLLAMA_URL = "http://localhost:11434"
OLLAMA_MODEL = "gpt-oss:20b-cloud"

SYSTEM_PROMPT = "You are a helpful assistant."
USER_PROMPT = "Hello! Are you running locally?"

try:
    start_time = time.time()

    response = requests.post(
        f"{OLLAMA_URL}/api/generate",
        json={
            "model": OLLAMA_MODEL,
            "prompt": USER_PROMPT,
            "system": SYSTEM_PROMPT,
            "stream": False
        },
        timeout=30
    )

    duration = (time.time() - start_time) * 1000

    if response.status_code == 200:
        data = response.json()

        print("\n========== Ollama Success ==========")
        print(f"Model: {OLLAMA_MODEL}")
        print(f"Response: {data['response']}")
        print(f"Duration: {duration:.2f} ms")
        print("====================================")

    else:
        print("\n========== Ollama Failed ==========")
        print(f"Status: {response.status_code}")
        print(f"Error: {response.text}")
        print("===================================")

except Exception as e:
    print("\n========== Ollama Error ==========")
    print(type(e).__name__, ":", str(e))
    print("==================================")

NameError: name 'print_result' is not defined

## 2. OpenAI Compatible Test (with API Key)
This mimics `src/ai/api-client.ts`. It uses the Chat Completions API with an Authorization header.

In [ ]:
# --- Configuration ---
API_BASE_URL = "https://api.openai.com/v1"
API_KEY = "YOUR_API_KEY_HERE"
API_MODEL = "gpt-3.5-turbo"
SYSTEM_PROMPT = "You are a helpful assistant."
USER_PROMPT = "Hello! Are you running via API key?"

try:
    start_time = time.time()
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {API_KEY}"
    }
    
    payload = {
        "model": API_MODEL,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_PROMPT}
        ],
        "temperature": 0.2,
        "stream": False
    }
    
    response = requests.post(
        f"{API_BASE_URL}/chat/completions",
        headers=headers,
        json=payload,
        timeout=30
    )
    
    if response.status_code == 200:
        data = response.json()
        content = data['choices'][0]['message']['content']
        duration = (time.time() - start_time) * 1000
        print_result("API Compatible Success", f"Response: {content}\nDuration: {duration:.2f}ms")
    else:
        print_result("API Compatible Failed", f"Status: {response.status_code}\nError: {response.text}")
except Exception as e:
    print_result("API Compatible Error", str(e))

## 3. Connection Test
Testing the connectivity check endpoints used in the 'Test Connection' button of the extension.

In [ ]:
def test_connections():
    # Test Ollama /api/tags
    try:
        res = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5)
        print(f"Ollama Connection: {'✅ Success' if res.ok else '❌ Failed'} (Status: {res.status_code})")
    except Exception as e:
        print(f"Ollama Connection: ❌ Error ({str(e)})")
    
    # Test OpenAI /models
    try:
        headers = {"Authorization": f"Bearer {API_KEY}"}
        res = requests.get(f"{API_BASE_URL}/models", headers=headers, timeout=5)
        print(f"API Connection: {'✅ Success' if res.ok else '❌ Failed'} (Status: {res.status_code})")
    except Exception as e:
        print(f"API Connection: ❌ Error ({str(e)})")

test_connections()